# X3D ⇄ MuJoCo: batched GPU rollouts of an experiment package (sketch, build 21f2a867878e)

**Status: a sketch.** It has been built and syntax-checked, never run on a GPU. Expect the doctor cell to find things to fix
on the first run (MJX field names, contact counting, the integrator). Tell a hosted model what it prints.

**What it does.** It reads an **experiment package** (the `.zip` the browser bridge or the MCP `simulate` tool writes:
`manifest.json`, `x3d/scene.x3d`, `mjcf/model.xml`). It sweeps that recipe (joint targets, obstacle position and radius,
link masses), runs every variant at once on the GPU with **MJX** (MuJoCo on JAX), and writes the results back **as experiment
packages**: one results package with `records/gpu-batch.json`, plus the most interesting variants as packages of their own.
Each of those opens in the browser bridge, where *Measure 5 s* replays it on MuJoCo's WASM build.

**How to run it.**
1. Runtime ▸ Change runtime type ▸ **G4 GPU** (signed in as the account that has Colab Pro and the G4).
2. Shift+Enter on each cell, top to bottom, with `PROFILE = "smoke"`. Every doctor line must be ✅.
3. Then `PROFILE = "core"`, and run again from *Configuration* down. `"full"` is the large sweep.
4. Results go to `My Drive/x3d.io/mujoco-bridge-runs/<run>-<profile>/` (small files only), which a hosted model can read.

**Cost.** Compute units are used while the GPU runtime is connected: Runtime ▸ Disconnect and delete runtime when it says DONE.

In [ ]:
# 1. What machine did we get?
import shutil, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi")
      else "No GPU. Runtime > Change runtime type > G4 GPU, then run this cell again.")

In [ ]:
#@title 2. Configuration { display-mode: "form" }
PROFILE = "smoke"  #@param ["smoke", "core", "full"]
RUN_NAME = "sweep1"  #@param {type:"string"}
BACKEND = "mjx"  #@param ["mjx", "warp"]
PACKAGE_PATH = ""  #@param {type:"string"}
TOP_K = 8  #@param {type:"integer"}
DRIVE_BASE = "x3d.io/mujoco-bridge-runs"  #@param {type:"string"}
MUJOCO_VERSION = "3.17.0"  # pinned: the browser bridge's WASM build, so packages replay on the same engine
import os, pathlib
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
ROOT = pathlib.Path("/content/mjbridge" if IN_COLAB else "/tmp/mjbridge")
ROOT.mkdir(parents=True, exist_ok=True)
print("profile", PROFILE, "| run", RUN_NAME, "| backend", BACKEND, "| package", PACKAGE_PATH or "(embedded sample)")

In [ ]:
# 3. Google Drive for results (and, if PACKAGE_PATH is a Drive path, for the input package)
DRIVE_OUT = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = pathlib.Path(f"/content/drive/MyDrive/{DRIVE_BASE}/{RUN_NAME}-{PROFILE}")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("results ->", DRIVE_OUT or "(local only)")

In [ ]:
# 4. Install MuJoCo + MJX at the bridge's pinned version, and JAX for the GPU (a few minutes the first time)
import subprocess, sys
pkgs = [f"mujoco=={MUJOCO_VERSION}", f"mujoco-mjx=={MUJOCO_VERSION}", "jax[cuda12]", "lxml"]
if BACKEND == "warp":
    pkgs.append("mujoco-warp")  # Conjecture: package name to confirm
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:]); print("✅ installed" if r.returncode == 0 else "❌ pip failed")

In [ ]:
# 5. Unpack the embedded code (build 21f2a867878e): rollouts.py and the bridge contract from x3d_mcp
import base64, gzip, json
BLOB = json.loads("{\"rollouts.py\": \"H4sIAAAAAAAC/61a+3PbuBH+XX8Fhp2pySvNyHndVamuzSW+a3KXR9Ncex1Vw1IiJNEmCZoPWzqP//d+uwD4kOQkfWQmFgksdheLfXwA6DjOd1G93MhY/PD+Z1GqNFVNXYmVKsVS5XW0rE+jMntwJuS2kGWSybwWRbS8jNayEu6b178IlYvXz395Jt40r9ULJf4elYVQRZ2oPEq9YDT664/nH1/82Rdv330U/zj/KD78/Fa8eyuek8BAnGcLGccQn+S1EllzoZYqXJRJvJbhumiCpNjlC7HYiUWTpHGYq1oulLoMip0/qtVa1htZipuk3ojtozjMlsVJNeRitRUuzafEhHxRRStZ7/C7lLkME+XhUQmwEpb/qJRRXIkoj8VNmdSYq9xiaLpjqs4Wp5Y7DJZFNfcuSnVTQSsjn3hQ85sX70dVkjVpVEtRK5WKppKwz/epupkIkteqevqtWESVROMyKQyHqMRkqScS1Y2UsPFKXEdlEuVYLvdCwX6GqPKFWlRYulSOClUltBTMo4zipEFvmuSXIouqSlYecVS5xGJnRZJiHTIVy1RgfsJtCvz8JPN1vcEgdWNfPLEuVVNg6Og6i0AUC3KE1nl4OXqWwJKwfqcVJijF+5fsWkQOKZAPKac3qkwhXNZlsqyIM+auyrh6ACc4XZCLBhcV5pFgKrBL1aQQZOzliyJtKpZYq6KzSsRtSSnUTd467cj95dFLmAMrHZFlsLIxFjhuXcAu3zKCtKxQZa2NJ4s02gkZLTdkMHi24zij0apUmQjDVVM3pQzDbkTLfzQybUtV7OwzJJbkA5VtoMnZZzjSxj7XcDMtJI5oSXnVrJS2qZWRN1mxo4nnhR6VbrPUksO4UhqVBzES2NCwlM8/vAnfv/StV4TaEWHnqKz237TTHeOqw8zy1NRQ5yipiURLTOEQmgQUIgGFTODr9nbZOTTt62j0G/G+VCvoW01ElfwKO0WlFOcVbBhRCDd5naS8xrFa1pThZJrC56IKa1eJqpZFxZ5fwfew5HA3Iv7hcTB6/+Hd969+Ov+rmIrbkcA/p8rUpXQm4tYxUYfnx75wbOiF2x1aHqKFwi7B82wcnJ3N0aBDj1rOgjE1sGS8PxmP73zNHs6/z/3s6QH7b4bsx6AgIfz36VDUOHjiC8jzxcOh0Ic9qasmTfekPnp4IJUV6Yt9QgLH3/SEP+a/X9+nwhk9HdfjbjQa/alzbP4r/qZDesJaavebiDhZ1tyglZ0gr1X1bJWqqJ6PuCOWK3Epd24l0xVnuropUqm5aE6I2lxwf6DZzpxe1nOg36CvlwSduQdNSQKnY5cSdthXzeccbsIDwaJdE55Zl6wLa2smNtc6IaP8gHiA26rGVA3j3EihlMQ0f1TNa1nubKITSYXHNIHRkIlRKvtR+qAfpAGlLBJUwI+tS8+MZnPuydFTzNr1141VEku0I4W7lJuCpLoqazf3PN27QV9eBCgrnODd03HwlNYYnkMjNRWKypBqTP1nRJlFWxduk4sHD/QAPSLTqvR9T6ujtgesnmjPwt89JTPPE4j0M81S7Q5GPiJP/X2rREZKuPcxMZqhyoHPTGtDUMmNsNpAEe7WFzv8lr64pFLVpvkARo6bZe0etlQorjLFoMMuBXaK+MEMOt7m/GxCau71XXkJlajEBDHckR76Lun1CTubOvOZYyGCM6c5cfi4W8jUTyQdRno8v5+BhhU8XI8p96RxRL2Bzj0aLMIyTQZhMyT8SlxyDkGe8Pb4cRR+Cb8+4XF+WMmAEEweuyYU3X70uPjxhoXONSaKWhMtvLl1DJNQwHVE5ej0f/7HwGqBHAANdbbJLrahQVquRTqTYTLxdSmbUCyghm+a/JKfYazH498/9dpk86HJPw3+DOzzRMo5T+O+Z5R4di3GZHSkPZJHMpILdAJ+3yK7VSJTAOrlJsrXsi2tmEzwhmROxFqqLIQvPuAHKt+EcE+slBMEmIp3IQkiicyeW5IcYZNETM1Knzw4Ya1PKI51X7WMaHoMTIkDoMMmAZBLDK7nII6g5BZU1SYqCN+RhA+8oJDIKZ1kGIRVltEOzWmyzi3jf9nl+FdgDcy/FuFF273XoEVrF4BrvT6NjXR26bBS23uxHbW5GgvTuoH2wVWCjVd4pRPdr7JUlevm7PXPsD7RZVirZrnpd+e2B6hn0M7Z1HItq9oiMk1EYIFo8JhHxAOSkcN0J9yKGWAzlWoO7DuVro0zLsS+dlvImFMg3961+TTxxTXlTwkLyRJ1rZtjl/I0w6CSNeIiwobAvQ6o2iMwZ3PPhnXitVzRiRnFW+JsBkPfrHJ7THnfNW33EDOQz8bzuUEALRnVJr0qwZsL9uCAVorgbYgCn+Rrd4idOTH1Es9GZbLjkV2EeZTJh0nsZn7XWL9bXODn3Xevwx/P/wGsRKOcjsk6RHgQl4CCxu2yshckcTelsCmwEGGqmJSCxtWplun8to2jZjiWfSWM4pKHVjKvVEmvM/viOnVSaJfikV2duDLDQHqR1+EVYlsP5L2ge8HUvC4XtCB6yzFzuBcJu2N0PWQUq9V/xycKAa2YDVy4ibABcFfO7cVdmCk8s/ZfwiZVvtgkbA/LJ1zWZVpSZpuRENSZuf+p7rOOW0ZYhtJg0dQhp1+gjbY3Hvf97CVQcb+3cx5sX2RN3T/K3aqEJ5EbxQDa5DDHBmCeN1EZa7KevO24p07M8oYUXF85CVtf4reAAccemc2+LaVpsMSdWNQ1LhPGAvBjKgV8jhGWPqfskHM4u3LvNVW92OWQIGibbQNbToKonnGYzClRWM7ecEzVG0OVpx1EK9mOK4ejFpke1dkAoyjW9IjWULoN+GMwC09Tp+qQGm1DakxyKDnpSbY2PRRuer5cfjfgsyoYoAMl+FwESNoafLrGCrWWnK6rXuGeLjJ/ULSni2TPDSy02XeBqsH/1Dd7vb1F17Bl+kkf0gx60qxECk433uPICQwc44Dy1gz1OeCS73JW8+bPxFWsu69l2uu+1t0HvBoQX1iManPLZcG41LVne+IKG8M281zG3HsV+ybleAdczSrE7SLQXKZxQD+0vp1elHc8vdSNd8QIXACWoEWFrI/YAkoUBjnwE4CArzEB7Tpo3OEQk0kIi7q8Jr61tXcvcZuXNH3sCfEbUSGhYf8VA1zqIz17RjfpHSRq0OsLw4EOeTNgjANJFv3EpoBRkpu1Ze5w5XINisiUVZO5cRAChqWBQUJBDAQj/jAVY08kK7GJqqiuSxfiHaZzPOztgCi64YcDD0Rau9Kgm40spUtPSQWU5XKfJ34Lr+GJfCvO5OlTQJ6aApaBiCrqgI4NjT14wH2uQ5oSc+x+kwz6dYvMP95BLy882cSzrH3xFmBv6FIEBDEBV6/3dsybSJSjseZHcHFAT2oUdEiyBFfacYU0fSDkFP+RgXKXHNRnvppFxPWUduz62O53tDsfHQ2O4xHcFzjq1S19GaKlXyTAF/ilU27XZKaeFAILFYEFrQsmR2gciNEzu669QKo4SRGirCakMdMMPU7juU7ZWQdE5+0p1D27dwbOpA7kzPdsEcX/Gdt2T/8JplXzeZ79Hf0DRgXDtk+xTz/Pvr/Bt+z7bZ9gX6/vZ28S8idGX/Xdx3hLLK+TpQzXXE7ZjVyGELBlr36t99zUbNdmJMPvbdD6DQg38zrYhnEbhA+U2WeOHY6l4ySim9wrAABs3xDY1fRscHhx67BKf4FnOROrHvIZafKRNHuL5k5P0/NCq1TZPsoTA1UcVt2QhZp1by4+EZBmWiYe7szxKrJ5cXDiYcLem1B9KMponUUTkSvUg2u6UxJcDjcqpZrRHfvz3eTz969o34/1WV7K3iG/3bP37zHNyYtwzW0F6eIFArO4kEu68Zno4oQtX8PvVKB6JzR8MKAPMy9uOogPX6V3moJghXNGLrSDTlO+4lrVaN2/qYCpYLNMl8AkR7SmdKqBCbUHu2WUoNq8VfUrKn50Qynj87Kkfdp3z1/8eP725fSEJnFCJshVzVcotczFTtbP6EJSnED9E8f7f51fvaBbZblCEcuXEnleT6d3ocbr4OmlXhZN2BK71xN77t8/0OKzczpDaE+xPurLVTG4XD2xJzvYjEepcPOoTq6luaQ2J3dPH3sc43QEZhyrNeSxO6oks0cwVpS+fkBo2Yb9Xf+1SVWemcKU//p0FMH7tKne0vsH9bmHbKa3Tr0rKDCcqMxOi9hBsOgUhbZrk63u6HKMrkxDSZcDWpAO7KQ98pktZ04Sm8S4JE+rZ85SpU2W0x43SPJYbl2HkNGk3dfPxnPnExkCHMy1LRWOrofudtKoqjlnaDoNyYju9Ay013gwus33ksVeShnKGHTO7/5frtp+1sDHt+SPdDOsbx5d/suXN74Y3vKYHQm5JXq7s1XJt5M6iOUyjUqEav/aWa1EtPeNBQsBshtcnZ+QFoIurSvVlEt5AkXrjdeFvFLkge31qmuuSh0MMeuGuRCsuJUBFSfn5fn3DjInf7QgJAMY8KCzsNJlGJtUnF0oCCXca+3rSyuK2j4LfV6XN7QRTqNsEUdiS18yFKVrrxEYv+fqCtn5/OtHZ602M+fn/jUbb0yc6yhtEAvE0T16G+d5PQY/9e/i7mcwvLLz9jXQQOEz8jXRofTPDe5BkcHg13So9NFesQ3HO8Lhky2XGNU6QdUcqdrP+nzeGcT2Z6wM4TbNqQYcrVL2MvBbObc9lbrz9PnsxGJHPI/n3p34EsIzIhw7x7T4AZt+BOnO6mEw5NAkx0Dm4FaNP1IIamVOUckzfYF6oGK8Tp0mT/BoD0H1kJM//JG+ckDaq6Dk1DkLxk5vyM8fvz/9xvnjt//MTwC6Kbg10l9eYuzxDw1cAvpMRvcaSPhEDYV5EnTrNBX710SU5DlCNKnJzz1Sc31klhEL3SUDwfe9p5hx4RzeI/GxiP7ixX7rgJ3Y2uafey+BSllZGqxLpJ85Ty12NfpspnqVA5j0Lprp0wOMyBawJwx2z4c4LlVNJKnuzqefkm4o38xurcWoTNmvRg5Ll8Z+bUGB3oM6wohc0eRcb69QDkGpuX2k4b12Gu8dolQ6O25Ju2rCxHtCDpArbXgpT7a7cqPygIxZ6d1/T7FjRIfyDBCmyxM7jJuY/O5zVyTzUbuVpazvVMBYWUT4YfsoPtWgplvO0+sxYvSrr8hHYCZeSpok7WVpIcl29Esrg5+7voMOPsAhp+xihI6J1l0cmFeOLnq7dYb+RLOll7sjUKjb0UdxJvkbwKlz3C0n7Q5+8EUag+zeh3P1BvDXfF3nVlJygMB/7ScdxpRHw62HSkv6cu9YVOlCPu0DiOH0u3CwQeDdb9dDYvPpojUj9LBGxOMXm/B5+wUJo92IPwFlGwXilflOjb7KvOejxgKeWYk3+vMp8URgX6Isqk/qzphw/vabLZeO2ybaVHtI3ky9/4UXU1s2dKQVu6scvhqRv9rTFaoA1MdnXm6/Lqxy15LuZVW/PwLbRZtmyYjRIpVALgOlqD2geVQuP8ZNVlT0sYe5eZwa+KMmQrW5qn8gqCjvcbs9Eayolb4b+DfiOyDLgysAAA==\", \"mujoco_bridge/__init__.py\": \"H4sIAAAAAAAC/12TwW7bMAyG73oKwpe1gJ006E7pOmDLMGBFswTFDsUuhWzRMTdbFCQ5qd9+lJIG6XwyafLjr59yURTPt9/gU/UZ1uMDrxhqT2aHJTRso9dNhNdbUw3jH254fjNblMD2VPohwHaKncQ1WUN2F2ZKfXnL0eB6HNBGHUlCbiF2KHQ+BPTgPEeOk0OBnCddiZTqyFbbVFB9zWJA5l6X0LIHvRNiWCpVnduWkMkplKjSfpgvEr+lHuHKY0MOYa97MllJCeuH1XcpHxz1OaPg8qmniBUZGUON7iHype6sNjXiU+aWMGhLLYZ4PRNJQbcYp2XCdNoLAw08rx/BaR/En9Po0HhEm+OaR2uk6PePLXjUJmRKI40vxMJJudNGjukA2hrAV4eekrmCbv6KKeEODp4inuMMstqFjpNDebAsOS374+wG9hRGOd1bBRwodqDlNE1HFqs0V9fiX88hiArHPmYiDcuTT/I6in94RKfVvBM6Pwm5g+hJJske2JsAZJOf/1l+Nvd4zyquJdzryH6a5/Zqv4DQaYdyw35yFI076NCjOIHNGDGtxSC0ngeZ4MYYSpEufoLrxx3ZAFYP4nM9nT+zV9lxSPcknbEVnm0ui2aqKAqltk+bX5vV5hHuoXj/MxTqHxmnpzdAAwAA\", \"mujoco_bridge/contract.py\": \"H4sIAAAAAAAC/51Z3VLbSBa+91P0ai6wE1lgwFnWM6QqQ8hsUpBQgdnKLsVqZKttN5HVQt0CG8q38wBb+4TzJPud7pYsGUOyw1Qkufv0+evzP57nXUw5G8lURyPdjfLZdo9l04USoyhhWS7HIuFMjg1EDhA234u7s+JajuT2TtDzWSZzzWM2zuWMaaAa5vJO8RxvEU940GqdyyIfGRw6L/R0wL7sve2eFh/kkWRnudSy+7MBZQadykfb9uh2STLQikVp3KptzWTME6wH7LeRnGXgMcz5SGT8NzYrlCbG4wJEiSEVzTg7/XD0jg0XmgOVouWW43NLMYfhs0EwYHye8VzMeAo00ehrNKEzOXQ05aOvkHS4AAhpgt8UUSL0otUGWq226RlaBkMtZaKCbGGQ4zRQTCKRqhpO0LGqA8Ka4jpBy/O8VssoNAzHhS5yHoZMzAgcikiljrSQqWq13Nos0lMLn+ErEcMS+Iw2Sqi0mIEfSJ9mDru9Rsdydebzp4tPR59OWi18vXt/cswOmdewD6/19vjdm19PLsLPx0fvzwjgocXw5zl78QbMnfWZV2RQ5wlPJ3qK9Z2g38diIu8ai/v9EvI0UsosvargqqV935KRQwVuDJ0HL5NKkDrw43InODjwicYreu7tXwFFHsWisOd7vaXfWlbcX7z5/MvxBbjHub0+nXh11fr5/ce37z/+ck7LhtqDJ2Kc9iB6YDj0gDTmY1r7WcaLX8u12TUtVSARbF3mMA9ateZOy7k0fHtDHPWW/iMSRuI1EiflmiVRgfwJEncyT+KgUuCKzie39Hd4mt2wxOqgDXrwYuYW0gn74/f/lH5UZ0GRqY7YhMvZJmEnMkpqPFxE+YTrdQ7UVBZJzPNtngzlHdMGSG3kBxzMcNpxg+ghk8SqqmRoteoQrbOleKpkHmiRwYeL0bTG3pF1gncS4WzFXgPyGzdSBVZLhWhftU4+nZ8fr+zNO67iDwCbgVkVQ8W1zxACWAStpjwHDCT/4/f/mggH8FuOoJIHnvMVo5ZFIlJoUG2rbMopGIkkQZjMI43IaCPxKMpUgVAPyK/qR/MCtiQRCr7FhGKxUNEw4bHP7oSeloyxSNsgS4iNaNAHk2myqDj4KFkKK7jlxCj7LCYiJqM+M7qg6JuOZY5rQ7gGmVESQfZ4UPIlTLRGsIRQKdDHizSa0blK1RWh9zMXUIkOzqWENhH3WNESWLDkAlTAPoFDFnNQy7Fd3UstuvqsNH1WhphtG0soGbFrKZAgnDGa/JDzKP6RSXCbgw5XdIF0ivFYVCCOQ0RhWWglYpuibOjFuSrllTKdp7iXqbQZsJ6akO5kHlu0ZA7jIkkYWOKT3FDtkusZikm0sLkrI5YVob5q/cA+0K+uQjri7OwtK9SjTITU6J3ySCH/sD5THmtTBna5f+VICnnYBx+zrtFJN4u7tz1ksTefT0MgRnLwbG52Xl8HI+eg2y+SyATPUAC+t/uibRUbiu5NKDqsy3aD/Rc3tzwJhdOM/fMMLorul1WcIJwmVHgU/nHDs8x6volO0P1IF5GWULbOkzxKJ8aZvxJQbzfYoW+KBKC4bLXCi+PTs5M3F5Tk2pRO20jJVG6EnYDyOm5iGwEi0pFHH84rQogZaj6D7jUP5rPE6wRkHlib67ZX6HH3wOu0Wi1EFnatQqTm9nzAxomMNKR9zeCbA2sAnveOzJj8HVBD2IgpPhKTyT9Et9H5KBeZxl2da4rEbQvVYbGEn7ehlJzqEpbLIo27AMngyRMYZCegMoNoiLGxIKoiAqHGIhWat+edQaXnPBKKs39EScGP81zmbc8ClRzlqIQE/AgiOXxzdnjIdmoYOKqYlHk7XgMCdwdCxrSjoaLPnw7Z7osX/b1HR6GRtgW3RCJciD1TYuzx7is6H7Gf8L3be4QizQITaXRoFB2Wbh0l7bnPilTcFPzwIi+4j0pVzA69rhMIwQlBF84HmutI1EjACMRYjJ5GEqgMdWLb4w6f42fsPRDmJX/YerlFEpB8oNJhr6E8xhMofau7tXwgOcu9ztJzdhO6m7r1WSKd8fhsKqrPJBryZECaMzZlVgelthDXUY1S1CUEQ5SrHQafIENobhFl35lmBdK0lVuzfgu9J9J+vSY+nrYfSG6YW4JXa0m3tAkFsMsH5w+J7Cx9Vv6ais7yat3QSj0Sb8TEI7GcQxlN3jq1wWsV6nMTX+gQNIMkpy8N7NWg7hFriiAw3GqRJdxqIuEpYfgLbPY5Z7G0BkzfSZj6JIFfbhbksrrSyx3Erm4vQDVrHlVwK6uWjr+6/8veGnC9UPI6Vw3BbSh2gsdipJ+RmLafDQO5a5nk8JqP9LpUEu5yG5CaV1Vk5wlq8juolUieogesjlzZh5i7ca3Is+ZINddI1yuBsvV9cMeXjgjFAOmolJ3HU0JlTxlNZhjb+x5hg5LKgM0X908YzkOFaEMLttprtmKV/Vhp6psdaoZ2TUt0UHVmxLndXGFs9nFrGOubFiM9/lr1dZsQ1nvATQyaLYOs57PdirWZXV5jbDOe1VYdj+XoEZ56q1mrO0jRtb6zpJAZvzWC9ghxdZzN6x6bGY8FaaPhv/XrgIs6+Ud/Kwy7V7bFLZ8rDPesPRZzsuUkSnkHAaCJcNURl9icPbt1o5Udg7h3UEdc7VfoXO+0dBGmOYppl9HBOPZ6VXNGswY5bg5fWLsc7HRM3qLOknoeLXRhinJ0FqZCHRXUu8RUdZrRTlXL5HDPRqCzL+stKTZtPrGuLBXNNpgXUBXZTtujDhtTXUh5KL9c3f3V5eq2rxqO167qw4aKA1N2IwJsKbQfhx70eUBK7fc9lk+G0eGWzzZskfVhZQXT63wL7au+ObwJbbX1/WghJR09ODAzFGNZHisR9noezo4t0AOey3LvIW2vqcuZyhWqlefo0RRKSyBn5r+g36eXJdPcsyTqMQq4S+An8TvJg1d9i9P+ruEyYeBbXDqt1JmzOvhTPK3LvP8NmetR9P+Ref+xyKvI950i728WeSNLnbKfQU0rxmg5Gv7vu3JkrezYmL80+svEJjAzifQ3DxarpnKTv2Pf1T+DZsXn2KiF+PrM5sEr5wvNAU69n3fDJjMBomh1TO09LaI8T8RImN6cghm16BTc7KlljeRIoncXKUKYMkRtT004/tUtMjbjGvR8dkcZ/wbNKs9Tin5Ecr5nxlT/rMNFc6G6prJEm+fGF97TWcQbRkoYjtGyLPz7TnhqJmb4de93F53wS4PZoUjB64QOlMNRky+VMtzb8dUqEfzAjirp7DwqNVMQNTWjFjM8qv9/gnZ9rE9tBbWm1pCuQy1DCEz10qYyHQ3Xwke+Q2tue4B6AM+apTVg71ElL8pquFRUSeJmMwnkFTeIWrsMgjYj0JXuhc1N5mqMiln7JvwC7o7YDVR89O9ub9V03/nPs3+zSllGK+om1+079oLdsZdonl/g30u2wHuB9z3e952nm/nUNm6mMe72nm1W3qdoxURck9SVmg2G79g2S7FgXwv7uqdXycQdiNUGAI3TXfzqzuk28O/ewKiGnM/IV97nDk1qevTYcY8rIqucjK59viQOlWVNEUH73gU+Qy1Cob7bJldDm/Q/rYpTjgwbAAA=\", \"mujoco_bridge/safety.py\": \"H4sIAAAAAAAC/71YbXPbNhL+zl+BsF+oVublepfMjRJlxi9y6jSyPY7aadrpsBAJSqhJggOAflM0c7/mftj9kttdgBKpKI775TxjGySA3cXus88uGIbhT5XVjbEiO5BV3Vi25FVWyGrBzJJrkbH5PbNLweZaZgvBrFKFYdHsh6vJ4exgenEyeR+XGZNVd5EWtRrEQXDVFMKM4DFvjGAnsxPDQDgTlZVWCsPmIldasJprAwpf0UCwW2mXbs09bDWqaKxU1TCohL1V+ppEgCxWKJ6hnSrPX7GU18zIB2Feteqm745PWS64bUAIWMctzPCM5RKMCpSm/aziVt4IlqpMvIJT5AW3gv16dslKUc6FNs4aDl7R6AqYRE1aLOAZ5BjQHtyibDi+1GwJGnBXWnBZxkEYhkGQa1WyJMkbtCRJmCxrpS0co1KW49FMEPh3WrSjB1mjoZvHQs6dIKlaAUdgjTm78AqKu7JoZ4TVQgTB9PCX5Jfp++To42zygY3Z98nz58/xl2YuD49/PHw7+cLsdDI9mlxtJl/0JmcXs8Ot2L/3dx5dnJw5gf7F28nFlNZ1hH/4Ifl5cjU7O3bSt7vPD6cTePPyn0GQYJDH4JM4VWUNzoh0+PrZycXx7OPl5NPrZ5Pz2dnsYzjEFWeDIAjA6RCQnyrDc3GGWI5+5kUjJlorPRgFDH4gIFdcGoD17VJUzCFeGo8ZAAfgArcT9LiB2LCohDBzQDUswylIAUgNdTuIKbpBJnKWEHJ1NGAHb5z3Y3D8Jb10irWA6Fe7cxHNuXkA+o1I2twYn/LCiCGrVOJxP57pBl4gapPMZu0CGCY3vJAZQcm/3YhdNguRoM52uRalAjXg0RJUmZ3XtTRODQkY+NPR4RLAV2TFnR0xYzX7RMlgwB4+F4V7N2YhnCzsOCGZFAL1bJxPx2ZNSzkM1o8obzGthpjWf9tkPkSEFzscMaREFBVsJcA7v1MkUIPMIUqyMpZXqSBrh85OH36/phAVTQ7YG9ZLku0qigkipYenPFzReddM3KVCZIatevvXTls42MiptZoL8AyqizOBPBOFjc0P/gXAFYhMMw6BLwueCr9NQET2mhuLqrt/8H82v/Uw5mVsBNfpMqL5jnMf0TnqRhc8UXDt2I9BmWkT0KsGMPPWaTuH3htjQN+A/LY11Or7jlnd7EO6hA2Apgj1DLfZ6/0PvqntNlU/3FeW3xGLMG5w+knnRb4Ahme3oigOAMOlyEZsBbvXcAiibBhDhn3zeKUasropCiywCkqMZplKG0rdIdtTw+IgQWnJ6cXV0dnJyeQ8mR2+RYpdhbJKiyYTALuwLpqFrHAE3hKVgSjgQ+tUHJcgrAjXrbiz95PkcDa7cqLQMFyE/7VcLG37UIh8M27qdpSp26odw8GrzZI5T6/DYQ+07U8IrLvMpMalGFNwjX8CjhfWj1MQBr5I7H0twFhHVyaFwFVJ+WeaR8ARI4eOr3ESBQHDWNxTh+G7CO818DYajG+FBkQK18k4R5qYXRHADDVBhCYQoJTdEBM+gOu2TAq/YD8q9ZAHWNMOyxfsGTBp2fypUhU+BjXa7kQLdyZWAq0CZbLXbvsbLxzLmiAQkQ5pEe09jkGkdtJKFGiIS6w+q6DLZdWI7ma3GsXvQ9/XWYnO8XrlxKzfdAtyJF0yGA9/yokW+aYL/UGHtuaQFWO2i91YAlogACnSDh6RW6CB+aB7FNj5RHvd5gY6wpWBpktkEewd/Pb89/WYqepLxyEU8RQAZLoGg+Y2K9GNG9ueaIvbukXnFw3wKj0fIhh80qRLkV4nlPUJ1uOIhpQ256oSm2R5L0tpAfx1XUg4D8/Bo8z1Z8TmLEoVlHdH6uKOpxZsgZRSaNW0eaeOFZRw47eIbNCt3aQzruYqu/elzTWTj/Mt7SKhq44AqGUqk3gfwJws0Gz0w2ordb3NPL9vIYCVnWJqWv+qXty/Zvh3r1qS+blWZLobAW2709xrjv+qBa2sNcMRw6FM9zuhpwetckCYQ/gykSWYdNHDqL2IxL/K+hT+Q0fKS0GcOsSebQRwRWbbvTMQcqib2DbeWNMAS53rFRKMqmu8xUHR+wPk/eE2wbWKfQvgaXjxLVONxTY9ai9XCC3kK99EbiHUq/lwmVNg2EO8EBbHERreq/A/ivunV/WpNNiAAoWn13gbcAdgK5Taq+htHwoqY/QcpRPEFp31qIbLfZJ9pwQeWfUl+l5tN7KgpUUYuA3Oj6u41vw+GnzuI7rcPsSqhg4TxQ/QFfmyzzrftBigYBnmLrutYZlr37EaWrxSEZG4Qnh8dQwBVnh7pshRa7MjGsGmDzaH5O3pcy4Lw4gotG4gWFiaCEC5v6bTFwNUgxdyiVuB2RWYsWjgihD39Nwu0fx02VTXbDSGE8YE75cvXvzj5U59az333dit/2zSN+SwZrAb1a9x9f4I+68OBqAFxZsi2OvDPVqjNhOPeLZJRvwuENM1YsgmF6fuqvskPH/Blq3DIS+Q6X3oCvxew+foGXxLMY+omx3sgt8XFzoCuckzC6Is8elDnffI382QKnZoZhR0XI1r27tO76vF4yec3NXQnVJb1CZt79rTE9V3eS9JHpBF+uZF/tuLM60Xps+j9KRwnAOfcWJGb+ueuwJZpixcisfMNGUkO/yCHZ7E1uEhxkQupLHtfQa86HY5D3Y+3jwNIVsGIjHeU5C5WMPhPbQV++pLR8+633Y8tD1HIesEcRdtagr7RO0GQQIeN7XjHBYY5wn3zWahsPAAtywgvgc30LrOwdPvPlycj4gQ3Oc4QM/CLrstBtYN0rZ7N2wbdfeES1r4Ud1gr12No+9TdNPE17+N2ne/s+9Y+N9//ycM/geY+81BUxUAAA==\", \"mujoco_bridge/scene_io.py\": \"H4sIAAAAAAAC/8VZ3XLbxhW+51Ns0IsCKQVJcabTkUeZsSW6kStZHkmOM2E5zBJYinAALLILSKJc3fYJ+oR9kn7n7IIESEn25KYaj0kAi7Pn7/v2nMMgCC6UTMXMZOm1EjZRpbJClqlQd5UyWaHKWlQy+U1eK/tS3JqsVqtrESa6rI1ManH3It0pmk860bt78X4UDwbvtamt0HNhIP+S5IrQmmTX7bTLO8W1jXizrKiw/HVTpnl/md+KFg4Ht1m9EBqCUpVnM2UkdEmz+VwZVSbqQPxy8l4UqsATmGCUyMp5jjWp4BelWEgDS5d4K5HVILRyruplPNPYVqVT0jMaiiTXNiuvRb1QIl/i284CD5Rh6TNdzMS1rER4tb8fYY2saeFgZvStxRpnB77YOstzbGjjQRAEg8Hc6EJMp/OmboyaTv1C2F7qWtaZLu1g4O9luv32yeqy/X6fVfMsV16Sc/XUR80veX9xfnV+dH76yJJ4FSi/9vT88nJ0OaR33pycjmC2LirIhxeSrFJDUcgymytbDxFtY9f33VUtzbWqH9vIObXd5kNJ1ydl1UBQ19FDoStVTn14W7F3RT4YnL36eXp5NHo3mh79+OriUhyK7/b2pnt7e4NXp6fnH0fH03fnxyO6/3kg8Bf8/OI4GIqAwkSfhaolfTZlVtMn5x59+ahNnp6Uc00XZ1iVylpeqrp3WRuEvHvnWDezHO+7vd7Jm+yaA9YKeg0Trg2ZRlc/Zeq20lnJQo8zeI3Wyvw0u17wvSsjSzvXpmDVFrJi1V5VlZJ4kqw2OkPimkzmvIW+o4+jZZ7BgYbfrBbIehaom2RxqUqr+cH7XJZqfXmJ1E8W9O3i/MPVCNIf2MHHo/dXP8KFf/1+cHR+MZqejc5ej9jZYdCGPqb8o1cBbo9XfGMXf0rmu4UGDGOELIgGg0Gq5p5EppXRNTIin+p5aLSuI7HzAwBhxL/EO6D3wBkYBFdA2CovpeMeQDvJgV1wSydCopSFAja1YQkC/sMLVS6zUiD67tWYgEai6anKgX5Bu8dgLBP24h05Fegvm2NpjFwOA9qDjAui2CK/a0usEQYdZuu+SH9GAc1lX0A06DwhZb1vKOmnZC2MnUpTTFnnEO47IN+wj9IsqVfeYVomErJNxaSSdtlCMPSk8BJ3IHF33/tBXPDuVnxuQevgOhS30pRIb/uw8hXsz1VJakTiB7EBvbW1RmZWddEcEuoEIk2k1KI9z4oMDGMJsOIfr1tfIAjIqxXCabOhQ23U6sCBquW1+ObQPdnc+ieZN2pkjEYoP6o83yEIwSWkxc9np3Dv7w3QlnqR8Lg9YH+O4VwkzuwTkDghznh4Mkf6aQFXw5SsRC4AliGCDP2GHKt+FlAIsrJRGzlF1rCMUvR4ayOFthw7D3C1CnkJkInPTtzDSxx++ZKTwsMkRchNVtU7c6PUOhw4/pJEVfXKH84nFU7BQ+RTEe6zA6akHGST9WSkrbWxYRR1LXFvudRg0viy/u6wRx1R0zmaKoVCgo7KUnxeSXkgJXVT2wz2kUFe946+pCFByispa1DzrL97rhOZwyRaFtsqz5CXD0E03tmf9NZRltPSONe3FOgewMFxETFLf4UqU/+8Mfkm8J+AxOhOJU0tcVyQPHUHt4L6d8BmujGJEs6EplauPFmbD19s25/CsJZajkdvgn5YDjYNTMlPnPdPqNqB0Dw4buCthAooiBaf04eO+BZA43TCKgxaSHE2hszFfahQlHyub2vwyO5nmeUay9cvLslJSlcNz6GsCT2beD1A4hRyUiV4zQLO/IEF5mZv3dBejsr5ldos1yoVeJkONkRapjbsfCXJUSTEn8TZG1cHiIXOU/qkQ+ft5fk7gj9d3y40bOJ9KJEpgdtTkzdSdwQ/Ea6tHoqrZeW+ouS1tOI5jusbhhoL/zHXw8WkzO+NJnpgndqdvXLIZj4csMPgcS4rhkyNnPWFc1l7ZuNlUHBbSfYXuPz0z6lqfM6ALolRSZ6i8mvj3c91szoc3HEV+h3dVQA6WuWfBUKlCdNO+lUE5DYf0u0E8LSwTqteMtAfFpOA8RxZUIdVxMRTUS6z7DWT+KCurXxczqQLDTpc6QG7bZ/8SbGQeR7eiMNDccPdj5zZ8IZXOB0CNC6B0+OG9GABXwQVQHzAuTHPSurSyoY6oc2TsYMsEjve87Ay4wDxUuZUldf1IpgM6Q6TYXuHjg7n/+BDZyXKsvb2aWd51JN6Jq3tynTXmxL57qY8d3Mbx9U6ZonzVUK+crxwPkOqJ7n6UVJH2fJCTbV3zgVUPzkmgy/Et1oH1ge14ni9oIhW4+8mdLH3HB76GjlAG1UQnukfctgdZndLIZt6oZlmKirm167UXkYwGQcVulQ2hLw4rhBHNFHjffy/F7/4frLtr433jUyzxsXAObHvuL8rjT7KLFvX+eXwWinLwJcIzzjsS7s5Mdtaukp1xQjuMnwyzm+p1briRTZ4GvyTJ/T9MhHPe1scoBS7Sx42Gdbj6XNLWQd9PjNI6cBZgkdtLb5qs3CvGPbAHbR1Oh6NXac+fjGhbo5bwuyeqFSnS4EkUBZ44XbPHUdueJMaVJ9wP2iAK42ZVebGdw5UQ75sS3bUJrBLVIulzRIcZBpHHmqpYPLQ7Vt8m84n5AFPUOx2w0IDH5r39GY5PHnxjb+fx0CJZ1oUbmGGgjrMYTvMeaxxuUeOdEcI7vTmR1QsEBPDsvA+pqs8s3Vb13JZSRnU7Xv75Uxby7Cgry5mvCJeadQzD19RfnRnIuF9Jytc8x3FKPQ505t6vvO3DeCFH8qMnh7zGl9msHCqCzp3o69J9N7OVNWQF0jQE1n//68rWo/36wk3RzgUW67tjzG2XDvw5YRVVHk/0a/z/6u29dmyhQxy4sbtzQkfF11m8y95dui/5G9OnnPHqkIk1HNLzkjGQdIdFqSZldfoEGNx4mBPBatKs9p30CkPqqi11IipKZhjcCjF3iuExsc8ujkHetSl8BO/D8v6Q8Zw0zvRs5a+PXpDWZmViIoFoGkyzfTS64dX9m8UmZ6iv/3WbQrlOZRgWM84bAxxMTNP4KmHnnOycdtjo5YWeRC+4h5ngBs4bFAZT3dQ+MAxJn1sJjHkVCvUFOFyq+FpnF3Er8y0K4L9yMN3tNH9Yfsjk/oD9km9oKEA9lWrqXiVN7bVZfdbRnrs6r9fWdtfycOJNCaDK2+UmSGBChFmdVuw0E8F2IMShWdT1Pk56zkDnenRS8d7WNBGzahriKeJPVxv3cstfGV+K5fQDYWecr2UH6f/2YpVrJOlSBYq+S1uvfFH4dpF4zcbnYf7iFYo7YBwvdYj15v6XMo6JuoDkX6tcDxKhroNd33VM1Odc6Qt2GcNQS/T8WtKhpNz38pw5vsfBeJfsuoNPkOsRereovhZPTl5P0WHf/rqanTMp8D9WuH7mLMYGbc98+XMSJuismH7KOwf1ShqaBpd1offDQUKEPpdQ9okyw7foK9QnSlSd5/NUXInQFsq9cll8yeKNljbAyOGFo/2XKLTfKuwYbQ1Mgl2g/aYp5AHcdy7ptONJ0vrWZA/lb9qxOJ2d9J4uPGNQYPWjq0d+ATP69yUyz7Wqm04pZVqd/24pBcpmP2HgnIxenV8NorrO/6FAgfCf//9n7PmrT7SHW4BpfaHzS18QxIEhIKtiJmmRVIBuL7Vr7VG7xr/swy2XEYwPkcFJ3o5QQGQbqrvWWBVqxJg6Ec41OO6nRnWul5WtE2bHRsj88eOw6e0+YsIO1SMywALeTrduavgRqLn3rEC3NE5zgU48Pk/mXOuhVUdAAA=\", \"mujoco_bridge/sim.py\": \"H4sIAAAAAAAC/61ZW3PbNhZ+169A2emETGladptuR1tlxk2dbtI4ycZOujsaDQuRkEWLBBWCtONq9d/3OwB4tewmO6sHmwQODs79RsdxzpOsSnkpGJfs7OWz5yzLY5GyXDLJy+RasLPqZf4sx3bMChHlRcw4K4uEpyyRrFwJtijyGyWKR8osH1wfMbXiGxGMRme8jFZCdcGYkJeJFIAGKrzmS5ZvRIG7cqmYq4roMKuu8ig/NIBBqejukcat8OZNQIcSJXNLUMXW4nZZ8EywZMkuQa/0fMbLkkcrfWuUy7LI01QUPsuuwmVe3PAi/vsIe5LhjQmCVKXYTIawtNg95DPFs00qAvZ7UhrsWBAjK6BrUSjw4FsBkry6CAkcdFdpyfhlISCUmxpLI8DR7yfnZ2xRJWnMNHNLccPev3oLVoUq1SH9DRdFEl+KsMxziGNzC6SblN+qRr5GUF4wchxnNFoWecbCcFmVVSHCkCXZJi9AgpR5aWQ+Gtm1FVerNFnUrxkvV/VzmYBPjSqGqdBbjah+9zXMn7kUDT6jxvpNVhmI5YrJjaXK7Ft+Ai0rHpU14pN3Z+HbX3y24YUCt7y4FOW+c4ovRXlbn4rSZBNKXlM7AJV8o1Z5c0WUZ5skFWF2FS1hqyf/Cs8vTt+esyk7Go/D8Xhs1k7O3r46pdXjJ1g8Yuzrnj0bp1BMwAButSWRTXP2hFmHYO6x/wTnjPUob/TL6fOT968uwosXZ6dv3l+EhPu7cTAejUaxWDIgTYRyM48dPGVpospZWeHgTJUwo4inKV+kYj6fjBh+UPKFSEUmSlwe5WmVSaUNFRqKtedYP+Nq4KzazM71XcxtfM0LyGwIc16VkweuB82zuQaUGZ5Tni1iBAafJRMr9gCuk8THpA038/WWB2JAMolQ5h/5hJ3+7bsjjYRc8YoiSsHlJeADeSVLz/BIP3h3KqQLojz2dMq+H7db9FsUgq+blY/ch2H6TIKwRJbABmThx02ueFzMruYIEe1ynC+bVZm5De3lm8UV/r35+WX48s2L1xc+u/K65LQYytuNoPNsOm1ZL1/mgMDDy9cX4fN3p6d9gsEI+xYidJcO0TXZyt3kT8ev5Yhgw6ckoDig7Rln37JjopDgr0Wq4a+HB2JzAPv1gXlzqUiV+EsSHiBgcPkDN9e3kkqTnkqrBxT640MKJUXeo5uTZxfvTy7evCPrGu1hLSqLuwSvpwnRS3uzteUM5EbiPkgEpoqXeRFqKDpzH49KSJUX9zP65IcHGYUwY/hZnGT3s3x++vrcMFzbsbk0JDtO5sNV4KLVfcIhCGJ5tl3v5gOVAhlsaE3cG0QI9Zy0q5led5hOpIs7fPadZ6XS3OBIhPUOXkJGSyRxRyalKAabKk8RRkO9NRuDlZ6sXEdIUVzeDg6ZRYCD3ubtqFZRIZD6JIhpcoNLglsaPhI8ESsgeW7jbxgB+YZCFlQPSfqsmrBlmvNSR2T9ZHQI1WataZA1pUkG0mPIu9VymvtslUCdA1gtPUA2gJZSkucKdGX8k0tnK8/rMlJZMjO+FmFbYRC9aiMiCCRBFv0Pe41c7DVJ4p057NYB3I3JTCghZUKaSsBmMi9g58CjJgjRUgAR7lAll+WWiN5hgRfZwSbempy8axIG3Q4u9T9IdutQXHQmzCE8zk7DrBMkJAMT4LBrYHwL49VSNWBTuzwZSsjtap+oCsIQFSEkn4Whq9IkEq5m32djD3+2DlhL7hDTvahmsnNZ1SOUrgGhs3kvCUiqJlSiz+LSytcZU6c5cvrKY1+R3mVFKwQM8budA5/gCNrEjH15OmFT5RUkapmQG7ifjKF+IivtBlAtD54owT7wtBKnRZEXFO8sI53iE5UtiEdqUiVbCLYlenbMoKe6bIHKlbm6bKUY7bQcXqOSpDi0CXhR8Ft31vENSzPoazxJ0yiAkep5SMPrCOvLVUeXD7VXc4cmg6xGK2WiIYMyJ9G7g5DxP/4czae5UReJ6Hqs77LGd/eYkXGNjhEZJ4EMu3Ws27F/veB4HauKVVtb9UojUxLPnCuqLJQz79vCVUKe1ZZeFOmOk5h09Xk1Df34lyFpk+/S2V7twiyHgJw+SoiHSPuJjckHePs4LKCSeG7cZW8J9Y8Xr38d1FB7PcCqgEkhIMhVAj0xIzD2CMV/laIYfuSzRyJd5DePjL8R1dj+ScM9NVw8QqyEB0HrB8B4eOQN2YpVwDcbIWPX3VNjEjP7qkyzDilA4w0+CudkVQjKfQ5tiTKhanvc27Ee59alLmHsex8I9O7Ky2IEOOFs3RkLfmNg640zZ4/RdWrjRIpiB3Ud+JEKgIMWMtaQtvSLeTc6Irngii0qCAu9nrelg1s7NbwYizSDoEdr2n7tgN5uGECIfMSEx48JP+AQPEooWcFXtw7FMHLaJZS/4NGaEGVcVpyixGyON+tvE+uYO4N/Txyt5FrmN7ITRhmZKfiBr39V7Jg7SI9NYiRLsZ2cna24n7J0wnQDRf2h0mUFZPNkPPabAYYGwKIDoruxu5POsUv/BhHOpO1Q958tZtOQo6wJla1esHyn8yQTjdIqFqEmcZHnpLPniKdCFzt0eVtDVJL9oRn4g21WtyqJlOEnYKa8UM1oyE6KXBWtRMatSx/kC7Se1xREbw+bWdEmrRSSkUjjTvN5N7Xqi7RH6eyqXxFJjtqXp6xp4TsVw13NGug6G3JJOMWloIDEZkewrQYNamLvHnK6Qm+p6q4ScQ/Q4fRgG2oYnCzRczdLlaWgUSaFZxSIJvc2q4gpR8fjYGyAqXfoDjfI/sxOJ7qfXf2Cgt7NzAasUAd7a2JNYquNs2Fk/SX54bfTf7cG3iuc1pQF/rKYaYZ727Z4b9CRE5JWlnkl4050bqnTc0Li8jd7huikpq4F7vBdIx7dbZf3oNSorLibXTsp7OyZcGVj4Z6SvX1r4d3PwZvryqyZFRmitS1NdefQN1BdUkYiSV3jSOiUjjx2yDoDLluCwL4LChXbRvgIlo2UaHxDeWBMkTVGvVUVC/KHsOCoehiijt526rlBW5UxPTpw6jFBs2ECsJ2N+WyDbji0ibd+BRWRX9erUx3HcRGs3WcXRWX0pRsvXTgR83wBmezvtnRXyR4CQOfoPdjaaeMY5OP94465yfE6F2gOXT0/mOoAa5LBdNzJ0kgpaY4W7V4xNJA9AKvywSEqPmy77Y0GdQOY6VcLRn6DgoFGFwB8OvTUhgAiqXu50RQJ18QnW20kJNFDwtWLAVoWdfxm39TmOx1e9zU7GQ4vjTSbyeeEma4fEZrlyyVb3JLMOQ3bwaUZ0Ps6JFYwoaTUYiCMNHIWQe+21usMzre56vjdfVAfRLoHypp1XSiiJXZsMoZoKHiTX1xTbWIWl3B9002Fdi5Bfj7f2VqxZ0Nk+jY3wBFgdMj4RSUjaJOiPGEPshwBMpdJRA4IKG13+wZXOih0tN8NQ312CXLA6UNxygR2PYilmDPqS8ay4mqIqcnjnvULrN1ptV10ok1jbCKMF1BLbRrnwS7CjN0d2HUTSlp57B/H4Voi7Rv6IKANU18zFC3q4XIML2kLrh7KrlaaeNW5wRrJ7OBoPoOBzKkH6tnIXef7PBv9fDv9/9rqfnnGlFXM0SzIN6XGSZq2RqzrRuSd5rhjikeq5x+oH522GnYiXAY5nxDl9YepAFW8W3+bCqoy8mAjObUcvDcscMwnkA/mK57TfMQIQ/thLwzvAP9Mn1D0UMl8KnXtp8C3t+Uql2yBToFaE69Lox6znK/48ZMfcNJ+cwuUXqA6LRCI2LFwnapcHvzoeF6wEp/i5FL0ZxtOWzgADdUXZp7QtEN+2w7R7qzut6CgvVhe9GaB9lQHUtcFzsTUB9RIWe2RqOk9ror6JLnMY1rt3qO/TJH1bJ2EZIY84aR8oUuCaKftCEthY0fdq7VlnlJ2ALCtaOwqoayLh46QkRTQneieUAcVQxVOUZp6ZlMkFgcps9mGTJ7T10jDUJvb9s6WHMQTnv7zvrrHRBtyJv2w69CpD2oqSZx7/O0Li6l99Dnt9N4ZDPMnsLl5F6se10+6lUOX2iaQUftcP+PUDWLsOcpNGWs5Uznu7o2RPvvekthMzrrtZzs0I9+eOVhzaEqB/90BuN4d/RdalydcuyAAAA==\", \"mujoco_bridge/snapshot.py\": \"H4sIAAAAAAAC/6Va63bbOJL+r6fAYn6YTEuM5KRnurW2+yS+dJz2JSfO9CSj1ehQJCjRpkiGpGwrWv3dB9hH3CfZrwrgRbKUpHvcpyMSLBQKhcJXF0BKefn2+Ex0jsTHFyfipdMV92E+dyORx26aT5NCWGmWBGGkxOzWCzpl8/NuW6jHVGXhTMWFG9lOq3WczFIQ5qKYKsFsH8JiKi7nb5PjpC3SJMe3EBw/47Erkky44k4tgsydKbst3NgXD1lYqLyVF3hxM5+E6oskVuJD5sZ5kGQzgTHFOPEXwi3ALBcPSRb5zLzNlDdTN1VMhZmE40i1JiqZiTBmsbgnjwiBpos89DDXwPXAyM3FpSpc3y1cLYwrZm4cBiovRBrNQdCaud40jFUnUyADaxEleS4ylSZZQSO4FYcbVYgYo/ji8YXfmc1vEy953nV6rKYk88PYxURFkERR8qAly0J/ooSXxEUGefpGb+KfnXlars8nemalUg8sDKbZypLCLcIkFsd9YT22F+0v9uiSeuDlS7uzsEcfhZumUQhhioQ0FRJ5rid5LD6L4391evhUccod8aocnpS3l2OipKgvwn0MeX0zhcWA0MpLZpgHhKuIFkzUbj1MQ28qNLnulwSG8HgRhbGvsrbIE+HBouZkNiSOZ77kIlaQN05gZdBHJRr0d14QU1c8K23xWV/cJmFc0Iy8Yu4WSYbHXMU5P5BG9QKDv7/AqoReLmBiCyxZy0yT1xsTisK8wLhuzkur0D1W97ClHIYdF9FC+FB6qnynJaVstYIMpjUaBfNinqnRSIQzNgU3jktVtlqmberm0ygcl6+3eRKXzzO3mJbPmdJMH2eRk7uP9P+8CKO8ZP15nhTKLYqsYqyNq3yL57N0QfLHqRFPfx8Z+zJk795ff7g+vr7YQuKUJljSzm5HRTKCHbdFuQymYVvv3A1UsSj7elPl3Y1mia+iUR5+wa7zojAdxbwBcy9TKh4RsLRaN1ev3t28uf4wgmhn5xen4lDIDciRrdGvH9CuB3Rmt8WvsM7W6C01LmEBVv3lLVkEHt5efRidvT89tftCBhhOtsVuwtevLi6IcOxGkWy3hP7bTX9zcX7CnPMo9L/O+s351a9MCgCZKLlqtVq+CsQosB77IogSt7Bpz+ZF1udxYV6Ep1gG0vo89jtFFqYpow7WeAybLLA1gCAZQwOZRDieA1Qcskxi8QitMGfr0eaGMKC2Q9Htl1ODucFyYyG7ukuOLoCzzOIhASDiZ1v3NYT5oN/ZHxKn3FGxnxMYWdLpSluoKFciL+d1b927Ub4+p3IwIR3asBYmf2/zDO4JPblD2R//jEIfzkcFITQEHm0GVPNIIIFm6Lct5rny0ayKAT4Nn6jxFTEOYbzzOPw8V6ytk9MzYbmx+Hh5Ic5POlF4pwCAdyq2BcAnvAcEsGmDJIzTucbySrFjFzMlRTn5fGxlcvCvV51/up0v3c7PoyGsQI6kFpZ8nJT2oP/ypyE9QyyLJdc69WmB5FLPcTVaEt+VHuIOn/b5CTCKJfdJQTzRaum29h4t7wwHzeWHQ9HjV+rruL5v+Wvr6Zf6ngXWM4gH48xZxU/VGIvLsxumqI2NVAuVWoxKULHKgeakPLJK6Ha8YPSHW4jYIbs1ftmVOjfNgqDR8YFjuQX8BYwTtrq5F4aHZzARpU0mJ40YiUur8XQEwohiAUHrmZhNeXl7SVBUTeqGEUhY8DNh7EVznxA/4CAGQ8DrT8I4t9slY2MUZtg2zS1m37oQBG7wHjOEJNXEGvhG0mjFF9miXkMGxhrRjHgODTNCj5EeqO6sHj2VFuJ30vppllEIlVNjY0O7IYyzJrACaVycl8wj8qhFNZsyTuuLJXissId5fnhmdpvYbfHjmvlwS2lC2WTsWrO2mLDGi3kaqQH50wGD0LCtwWioZYXPw7wZLx2KMUZoCP3BZGj4gxfpxUEzMx7ggXGHOh4BwzTemM5MMRk2RdOjWp62Fo+shan6L4alKBY3vBhWoMO8cgogeR5tEwT0hR96jC6M8BXEiP8WVwg5K2siZCEWCmuMuBl9EX8uENS6FDLoABEbJKItVAW0Wj6OUSfEnvCpoyMp97GB5gC6fF1jxSJVpLB2qQVapFIL0FRBWA+PCQ/06+n15ejm3ZvT96dPsD/YO7hJKaKD7fjhPD+USwBzPugO7ZV8frS3i93r6499If6CwCaCk8bQ+Xr4GcyjSOh2PSueD9GIL22xsLfI8Tp55B4kwr012BfPBMnRFvppv3rqDb8h3PGni/Ork9P322ZbRp/b5iumKpxMC91WjvWNoRC3/P3iqV7JNEg/tNuQnfgbYGiVoa74QRQPich5DXJ71zCnFxfn726uz0/+8EBkcmjVA+zk/+7i1dUfn0SBcAZJ1eNOtpenN2/+MNdz8pHK/5CFbjyJFJIpzZ99qgn1GmO8OTs/vTjh4CoIVeSbQKxp+idnHKb5gVxhrxQWdlMgaQMtC+AeM9d7fRBI3ov0TSxpwJXBzBh5AJycHEICTcucvknfFl0ba9xb83U3v52/kyXsEDy4wANPESIhi4nSqbvugAE28PZ7+AVspMhUYm+hjbQnOqbDSu7RCvCLOBA9jZDrThbm/6oa7ejgElkolBwB4IIAIcJxEiWZ3n2QBByXRbZ6fnTwvNFpbw0tkQWsYeVWbKxnhVk0/ERpNEbktcjULIckGFk05l0wVsI/w9VuWY5v0q8tx4ZNlvI0QHaCSDgllCWn8+KpeJo5NgET5kxVjiWsaej7CBFg3toJI/eF6tx5VCAYUg8qs1n+/6rY1qz1bP5t9t+cLoyPFmW7Df4Zz8P+k1ju8KbVqhvCQ7Mb+l9bEU2LvJ/2Vm1YHI1TSYdC4QMu/RwtMZXVkjusDp7rNr0HVPRd8N0WUxLf+IS2fuoN7c0RLfKc9ZBb/Er2NYdSyra3tvj0B751vYstOeKs+1B2xXK6El15tDbwpvvOmuwPnle8/uBQnX9vLHu3zrf4snIVm/Kw4zKxgI4DdAxQ+v+vStb7qg62irXhAnNKoGlcNkD6PSrDzl7X6dZ0C6brGbreLrqSC+EcZ4NEug56TWQJ4yCMQ+RYaeTGCJz8zH2IKd7f7wro5vMc6JY/gb/v6/UEFbapf9McMJdu74lBbIvZEOIR8b4J1xbrBv+dq7EeOcxCfx2DqMraSBjo795ti/u4IpupfDoiJ+z62QD9aS9vfInnM/2l4hGAR7DBI3A9tZ0HfXnKgzgTYsap4+ZulrmL5piDe7d/70L79/HauJ7a3oe+DAK3H1CfAH0crCCjaqdX9095xLq0cm9VZTsrtXWSkXKVhYT7JpY9jcBEnkShfygDSsGlLsBgzasRdW2jsEIzWEiD8axop25FnroSLlIqlYEdpqFN5akANWCWyJIrTkDSzJ3M3D4Vi72EirUdQTXbBTKT2KfIl2Mz7VRy4Y5Bs9PVIByltLNSnTE2NNepaUKZ60YltCTE/ioqSipz5Lyr1rYS7RMw5HjtB9J8xcxEXklRxXLjnF4pRwWo9FSnt19GdWt5OJxfvW+XPO7qaElru2rutjLipIpEKXhZJmlXZzH8SsYEmCjCImo0cHRHyXCjSMlxrj7vKaiQXp0klYVbR7w3ul9yEbmsrOeYNH7KQ5ZVlerqkoMJ9jeLKBrmmIRKbUYPZSFlswJkCm3AimaV5QTvzWoGNF1NvlE8a1SaWZr90Ne92o0S9PX4Fj/Xr9+Ofjv9VCvRbsL+HcLxDZTfUqcpu4plVSO3Knb/kem8IqDCrKy51xICFVRBM/vN9Cll1Sdad0ZRFT0s4sHN/CaVIWlzbbHR3siPNmshlI6tqgpjXYallcNCreVtkTsb+y7Fk2G/IUjo7/NcMS59snXVlHZ2nHzGvj7924teq2XAKlQ5mS24DXS4yRHyKJnTptS+ltBnzPUeQAfYOjFtBLvWvw6qwWGiay9rtPTR3kgCqL/2NeSjxsMadV2N9IO0BlhrV2JUjcXj8xjpcF0onscP0JSKLebdAGqjRK2qLeb3+voE9jdm9VmSj0UljTHmkr+GjUDSVJbjVcOC9NkZTcLayfvt9fnVh7a4pWj47Qfeglo9t3Ghs4Fb8ozyF2nbT5Ber8jtmpbRr6Fk4mJ0fPtExzwFOp8j89cKL9+06Osbi06vwniuGpnahE95KUWY3To0jtPYYaxXu1E0nykGiwEcVHWaq8vefJIYxio7oyrDoeTit+QVOZTm+MuMdg7d8+fDZVXxtqjIbr7bNtLqvfVJrs+DxHAoK4t9qyHKSTKnI6CvijJz83x0Nykl4NRjxhMf0afB2FS01jBqmy53yPA96tDspNjbYg5lALBVQc8GgWwsT6zhwJrHfJxuy1WfSjZsUjEAg6xKjzXUOrXXFhIYpMPa5sH8puAz863SnznQhMLk0VKaAIeobLjTBqdGWONzfFoeW8nX1LetNy1+9CGVvW7Th09C/xrfGlqvHfvJ6RkWEyMhoFqa2a0q+TRerLbkYFW0tDYYQpC2CWbqeMd3HinWGdNu3gxx8I3Dm/FwfY+b0tSfCWL+5OzFjngK2eh3K4aH/os4pUiRqpgTrkBW1178ROlig4dgfKGDFYSSmUuXH0CKOKe6ClBCXOSOyWPSmSHA1GwBYd3B1Oh0w4PYSiFCSAtKyEqrE0kcLahQY2CxDfSu7jHo5rnd3r6NLGmuOWg6/cIccrpIY1rxyG0FdJrEplW/7GassNZRWCxot8Chu7yduaf6zNwiqmtgelTLDULE6hz4cSsVatjgdQdu2z2Shz2SuYY5Xoysj3SnAvwDrDOd+uhCpZdEyRxRdUNpIN3NvAygDHu82sN+cx/GW7NvYBCvpg666Egc0VVc6FMczJPzbnJe8VrxWJb3SzaunSB3osQkSflOSHn5pLxjQ8ePpLv34ST0CTne8c0kmAj2ASxTFwl71UAjOlUjF7WUD1O3kH1xB9/rITKk5/sVGyOa+FQdBgGxLVMjgCnMcsu2h9r+PZpTRle5CqqUrmc+ZPKOprDLEzPTqukZmnumwqEWFJ0MND0Xa8xjb4gsp+v8LJ6ZYaov+/Rl39mvvmh/mIaFRxW4Dt2LcbBD4n0L/dtEapclRnw3N2ocvO3/+FeK9h0Ve4haLTkvgs5PiEWcqXr0wwmyCxOJItmgiLVacpkCmQD2ERRX3olpcyudAqNx81oKPqp4gh2Nb9XR6tLETCNK8HOCzNGqvj8iZA6T9ajHUmph8YgH8NJ5TL++FWPVyY+9arCgcxJmUBr0WqcqTaAVoWQWrOmWHYgG5RUQtopHMgnfoW+D/l9fDofNMdx5MU0y7HkeqDRlvJh5StYMbwSGXvpCe6HJw6tvtzEXrRmi5ItsdDZKud/D4+KLIG+isjjUmCRhdUT3qUlH98Y6XAGo/Esu2ztjCjl285Alro8i/+9//rdxFW5D2DmhGsnJOYLsm4BUZwyQidOBupmTAzRzsP6PjMrwpIUqdqcet7wLTQcG9G3iynheU815KI3edatB86bAeheDpIIB/VFnYLSBa5vg1F1HT8ZMdZJcujhzSeiFvkci5cEv2EHCGPChxL6WgvcTEOhQ/v3DGXbUL0etA0Iqsz8O5TltZG9KYb2s+yL7l6CcKtc/OpjHIXnRQk2SbHEokdpMimkVatE6S3NaRp3PgJl0BMXl242ubAdlT6rzuvH3dqW4t+x5FyIIx27Z2ZczAE3LSmRCwqdmrFpQtNkknShElOS0t5IHZN6jmZc2iy/CWm7iy6otvgIqtuRRn7NmWwc3AFKF339QsnceB4mu12yICXsKks0wW/6+cc83Cej6asNQHHGycVnSfP5PZPb68khZutkrr0s6wNwVBGpG2xy3vebs6NjcLJTfDMGXJRivZJOdTjwaHC+NBPJbGVFJtjXnaNw4AuXWO0ecXLTWo3+8VprHy5V7H04YovRSIC0+3JOnH19dnl+dSiFfXX1CQExLF+kjoSKDhMT1tevdTfjKncjvFuYQ1unu/yic7kv6528/Mt3voXrgSqlWwZtkpqhJCl/lXhamOh6+hr3cc3N511fHyPDQdMMBCF8COOxddPEfJYjsdem7ds7Xwfu1eN7EASuW4yTMlEff3OiC4z2/bDiUHeel6PREx3lBRWJsA4iADdh1fqKuS2+aWb2urSPzOuK3V1CmNueD50CYo9bG7bClcQ9cQazXsy94wXYCY+2CqdjISD9eaN9ENRbKNTZjhrZ49gxMB6V3GK5Wrf8HeGzPRygvAAA=\", \"mujoco_bridge/data/contact_arm_template.xml\": \"H4sIAAAAAAAC/5VV23bbIBB871dweI8lK5JsnyP7V3ywhGwSEBRQU/fru4tuWGkb9yGKFi8zO7MLqlT/pmtNlG64PNJad57V/sysoqdvhFS1VkZIbgnrrpIfqWWNYB1Nwo/aeKE74oXiznNzpOkmTTNKROf51TKv7ZEKZaSohW+Z85TA6g/h75BJUvJy2Oy3I1TDW9ZLf6reNOwmDQPa7oqIWwCEcpjvLQ8MEEuhhOfNkXrbc0osFAe/vWSbnMAfQFZXrhVpraixQty2J7B1i490S4nT0vI2wG0LgkVUyVQClvOhrWwuurljBHGA65gCllZqbSnxdwOBkayDApz4BUFGMsAHdHu9sFD6K8YHeGQ5GZU+ghnegHVMTnj1XYqu4YBvtBtcAoRyYpggy4UjK2ABF1/LP1PoCxDUkk8Uzty45RPBZo/GFCUC5BFNpOKwwwykyHYRBbozUvTGrEoGrCENEoeODpnupnsZ9A3F3KDJUAv7KcatM/6jiMBwBm/eZ6eYcT2qaq1WXo/EWGmB/xYpaV5QopgLassicg67VaL8A9ASHP2AnIZ31raiG0Z1XdK6VTNTmLEioiixOXt8HPKnOSJjpf5YjJ2UTYkra7m86I9nfH10NlA852y+dvY1cjaPVE+iw0g+JfpLa4OTaWTtbnQVbNn9L8ug3AvzlzPxSWi56Ex30cnAErDpu3JlsIPraaE5Y/gP9CLqfhLdOvN7lTxcSBVc0T1er2Oe0vC+OmHnsEhJGJH44NXeytX9iUvTHbon+/mIx7hhvFag48h9gViQUV+VxHVXjndulhAQwaC1ivDZ+awiOa13DPU9pg/1TWpaC4nLBuwLRAI/EJToy9s4CqFVEK66N6F43de3CCLE2Ez/KbtKFonVO7+HAk74Nu6/aQVc38NcvITB2MIDxS05xnLnpiQ4Ujj2ZfhezYjfqmT4ip9+A7b1u3LOBwAA\", \"sample/contact-arm-experiment.zip\": \"H4sIAAAAAAAC/31YdVTV3bY9h+5GukG6U1LxIB2SB2kByQPSId1Io4cu6e7ubjzS3Uh3exB8eu+77/t0vO/O31hj7N8ae+/5z5pr7z3VFBGRMAAAABqgVuyNAT0GGE8TEQBgRwMAsH9mISZ2Vm/MnZy5rJ3s7cqgY3ZTPHh+J97vX6rQmrsVLWRj2BHh0WKE0YTTfBqmUCVsiGaew1mu3a/D9XOXzgiqSjdza8l7u1bNDou0kIQZcdUDqvvjCwKcrwHVfdFpU0foNBY3+Yu6Kwcx0P4srkH3l7fhlVels684NAk/knjBUwUPFvOf3REo77HXNwkvLx4Qp3yeIWp5VyKME+z00st3zIfo5Fo9RSy6KPEpMDzt/D3WSPsl3xr73d23YauTh30riwYpL6rcZGL1Ota5a1ciw/MTc6UAknqQFHznToLpwu5KhW8c/r70fJlPt5o/W1vmNIWqnnvXL0RA8BGVI9zz+HDz8+YzrQ+gexAbTcV69nixGrU6XTfNdNTE3rYcGE+3KPJ0N2FdIy8wEfah3p9cQnZs3XY8vachAVJmwmyHNjqGbytVtvA+2JHQRGEsjnU9+tppXdOstWaw5fv343G6781woJcojHl5OSdGsbWJT0mnmTfY9EwzcohTIMo7Nuoq1p1SIEbOlWZG8EpeTZ/faOzBs/w5hNXlMC3dvKj+WybP69XlsDNHgw885ck/SCL5q1+pDGh3fD5v1hEnfFFeTPhlu07wsZ3ALnJ+5JXCfVBqEtdUix9B0YPehAKBZ+CFd1pKKquBTFmCpUXhjjOKVecS56iSbk1Bxpx7yFRJOE1WYvPKHmRXPCD6cKpke1BDT5NCVUSO/Zh1/vUmuvASvGBfWGtHk04gd3e+0WaCfHyLkguV6V31mqSOC/gR+Yw2V6Pr8Zcd48HANVGyWtueebjL1cVpX9ek8QMVJ+ybc9J80rDKhUi2C/+s5qApjsLBjMcIBZBAkCjbgRNXQi+jOfQsps2qjNIeoIaT6/i6/OW56Mt28s6n7qfE0s6UNEO6CHzNFOiG+Kcp6qu8ft16WmxkIQfKo+8xo6OHHIswsUwwOG+Pzrs6+7LXaEhMPKruTHq6k+C8Xy6a6XxaIh1uM9Dt6kQtwpmvu7MfTIWCZ5XQCJ6kmF+B6UxGKy8dSVEwQ07GOLz5aGu/orNuhgqhXHdHvBoMfapkY+KyiBgyvZJ6GdQNrFthDT9zYYtbnNpNmzlTPsGazaPPDNiYIz+rMTZ3pFhYb5wmJ0VXpMhR6eYZCHvlrKYJThMCzis//+G93PwdW+1/5Qf4l/wAfwPSz3DnN+NW+02g+an501/RAQBf2n8L9NcMJ1NzO3Oun6M23QPHxX7ih5OCua6Y5mT8Z0vA/ly+eq5wfcVrsOPB1NVb7c54MGYnfaNcNv9DKurgpQG8I2aCee9JBOaz0/2ZDyIETMYwlw6JHbGSM9wYB9CxboJTnBKRwTRRN4ERkniy6kc34R1E1W/rt0O3bR4KE0qj7r2Ph2XUKyPhyB0Ww+ozCLpgMj5yypiRIJLeAuTUCFvel/QJoxFqdOhRzyTfyKDEt8tZrzupPebjqcCaBVnxxYAZksf9JrfUxpO2PRg8Bhh4YZatWo/Ssq8WAkhefjXqin+dbXz649tR5la1F6sK0rebjsGLajSZF/YMj3X8qaVrGgseGzdlntUCFKnMLWJYR9yxpCYL8an9TtWYG8hwt5WYZpGVEVKQkbBXL1qDoB7Sk+JExszocrHjT9bHuMIYfV/MMXxP6OxekGeT2OoLEA/ioKsFiNFhal46sNzsDt0uWKCL+Uj3+xLqEukz8j8z2yX8akZGFJL6vjS2q3rGA/kQ1rtV5Pi+/RYzJTbgjZN2rPRTsAOWTzGoPQjE5PSiF4Ij+gydwy9zDZ3JqnGwCdewxo+EzcOwwC+InhsNJt8N7kRg7drkC0jXIs5utcrlmQcGByU+GdH54IEUTi+gzAAOY7X+8g5EkAxydLhPcVYrY1PlJUys67dB27o89MpujcUxkeSZnZZUz6g8TKthiQUnIGB/ph4zxkvMDOIOSSnu8aelypNBGSguuYl/7NV/ju15YqVZXxmzaH5+Bvo2yZ3HknEPj9knlGbpNNb8Afe+b9pXuD+1ZA00uQFpq899GEyCTWqvzf7QVU1GCDEcCvgiZ1UMEa5gOKt8WKjWyeVeN+Y2Ei2pLuEu2zZA7KgDgD7j045d4Kbse8iuxSGuKh2YUkh+zKjOAc92eFmtaA2eWPJA8FxfzV/zXEehYs4ZH++0XWpZuEMGwNFaOQe29+gix5zRMp3Z/CKa3tNvA9blniaX1hNKgdLOC0vqo0ZI6ANVAANcJ+8MgSUF308C+Ziqm3dkt9YdzpK/q6uXKoJzPKkRRN9NSLnl7THAUKuvCZ82XJZXrCdst9KsZNLEPJLN7Eg1lO56SO+0z4xBiTjJP7mLWz5H0tNnf/gUHnHThoBWQuAscQBMvdJux+vaWA6BnhWsdCvvz9BWY2Ikg8UcGoxbsR0teM5jKYlGS7DZZwPmc49j0LeBLQkLYrOLDJ4G3N/RRScXzusq6wuXH5G5xsOpNN17DV1K4ecUAWP4y5LJ/Pg09f5Oz3LaxlcG453BpZ3swyzG87EyDRufjjfvJ+vZz5tXZx2elxC9rEV5j/2JezYeStDMAFRDcIorQlYtCGa5f2tPb83mxMXna09fb3Xve7Y0kTZ893g4ZqQdujNCcPfpwQRVY3RkMY1YMBG4C7D3E6wkfqrBbZSRNstIolCxcKFiN0sfcI5K6Z5UQBfoqOhOBH3CkxYg+H0ctVlX+iwY6vaU+wzx4wt82Y3IKY61J+rdAAwJ7m4mbBQEUj564pFYugIQ43j2ySAU4BjIEGRmYv8tDRXsxGbHPLSkZObHlvlJUCo7Nnztrl/CPejNq5lFgQmmk6PFlrC4vur3mgx1oCP8H/rjI/RBstjEjBVdRgQ8PCldB643lvECZbuW8hYUCLylpAGMu5lIlRfbtMCIfknFQAuQ3yTZnqvjkFcqU12oAHM2hyM1DxljhTisOAdZ/jjU1QCGAj1ikk0ck8BkO0KD6pLJONr5yaNF4VZey/XgG+Al4X5vIZKAUWDSdGhm8gg6hx8WsTiQx4Kul+M/H5lUUW0IR+mxRSzZh8J1FEUbZb1DRcoLG1yLfGe/DD33GnCZbPGQbaiZvUJ0bih10Wqoe36luJRX3Hsy+VBU44yBdwTmgSDQHfJMlqW/G63Nc/KcfsWjtfhFf3JcYOSKUuNmlKq+3MMLsXDZNJ8aEdgyiaCtejGTlBct+uNJORckKjQ9XOVh5XPz7SJx245TUXmKDwlrYNz14f3FSEQJf5JAptF9qTiTCJesF3X3paHZWoTm9wC424XSx7cb5y194WGnj59NY6UWpgSvLmFQbHY4XtOqahHvqNK/uZWjS7/zD0ewW+4iU4HWZ/gJpe7uo7XrI4dYrnyMDM8mxH7AxvNBeuNeiF1hYS9zoV2HhsV1urC02xklJzznS9ilJK7PqS9hhDX9QUEYnl1SX9LEMCBA6eYcplNAWUzYnj3OPsop8BHeBSLr3PA3JOtxhPqRROxnoDDa86VszVHK+UvJKCIiak1Cbu3mhs9uRXNZxailQj8o8h8yDGwqUr4QqceUTIU+sXGDgDTxNLE9xCag7nTU8jjKCJ1YetZ4GuWFWrh6qq6Nef7WoZZGO7HrQ2ZFCZ+esptLGkvarKyb0pf0FJcwu+gwa9EEQEVzuc2I32pNivRe7dtlk5l99GERwX/OcSQXAmME0L5YFJJ4ZybabLFYbgJphc+5vVNf9ZN5HY19k82prQp1xdLIkCYVBluVMyXvT50JL3UcdMHpuy6JQe8VHmkkZkFa8sjLo4Bl/WEvr2nb30m1jOV55l7ekAJ3OlU8j+HUPezfaQyDxP29vjamXwP1cPIsmnZcbJZb97ZsTncb15jdBUSRXFNm/dPL3cQ74S247tU21/4FCwkmHqK1BLBx2RCJ4liRL2u71tFFUfrP3SBjE+KmhCIGPVR75abuIZJ8EnK3DQP5thupVawMC/7KrJ6kQlH2IWMpabZd+UedPhRqgZdHVxbIqajr3fpd+VdRlXsBmQy31WPPvd4econoVDYxnGlDbxDPPXsmIbrTdDbUQmTdrzN2eOC196jfAkKx33AG2Tz55II4Z9iPeyR7T95eFTwSRMtubMAhnMjiXrUa6HQlHHfVtJvcSt1Jh7pdWkt9Ynw6BO61xBlVGX3DM/o6RSJmrY/dk+Riayzei3Quc53OW19ySktud9BQjJ/dR5uE1uarmdXu7SLUf/qORRezHaGfgaXqEaWYd20VFZhAZJlhx7ZnWAHBxwyhQpNEAZMOUvf4xxlpcUgcNppoUUUci8qID8uPjaBEA1hYaxK5ZlizxdYG99YjU5yWNQcD2AT2JL+SJXFKzibO4iMB1Nlp7CzGTrUZP0GAJ7aRzWC1n7Neatx0G1ivZ9pE0uv5qOCNv1GEUGkyaFU/nGNI3MyMdQy8wa1mnO5dCn08KUqkM4vbetoDpavxzI0Njb1bl2dpjOFn2ypO/OH5jWnsWOgUCXb1ImMzkmOkVmLW2irA3CYFZ0qlxsSGErN4dzYXEwNbqOfKed6Rxa/0SDW6Ksq3h59MdPvr0YGbsIVZQvitAlKpkd2ZR1SsypyGCOtQGpAFo4ptOkBzSv2tej7VkHXa/uIhNew55zVOH847vLLvJRfutyviEg/c/3xjRP71YLM2ffPHlZFmRrO2BwEAWEMFAHD+MwVib2Zuy+UOsYVqLSZk8+B5ZVYMDrUEoBXuZHVshOMJ8Pubfagsv7tapLKLS152oAn5ZFpQXNJM+D7/nU4efVRxo/iEel009HhU6bmqfRBRnnVt7BLejCGTlHoPW9cr8m80UFfput7wEL2Bq7j+AxEVFvyyTZ2eND5ZNnJOGVb9sdIcJG1fjBs/JYIP0yZtq1hqxBW6zMtF7jqVqZpYlovs9HYAvGGTcv4l/qccMgyUFdFTydEar7NEbiHQUg0EX4CufpO1HAQUTrxRL+FfRuHVS6FQGh8yS233AZ8jO6yfoDKtha+6uXDfH7+Xutwcu3JKnfV5+DtLtFozTiLjPns0gniAZFbqQfC3iQZSmrW4XDM1OFa+B/IM75MNKC7rwnvEdu16N3S88ziToFGLaMZAZP2lbSONQky8j/ExRNqrjesMXZ4x/WZPEHKBylkblT1YuXKuQORmfjXnV9wKKKWS2AR30dfJ1/mie1dXfUtZQ2qYd0whZYQPNWiwtyj2ko8LkuqpioujrNPIw6omsH6MqW8I5OSazJSM1XQ+OdSOPsd9qvj2UsLxcFXXIKKeZtpcfkRbWcmnvzrZbnu4SvGdF+VE4hBCaF4zDCeGYKA3eEDfcOqeODCT85qoQjeNi3MF5VGB//7L2syxRJDvQZUb38Pw5rIx923NnVHzCgLzpZHKEHPfNw6RFWQLZMVO3rmvmAWc+mD+MbIe3yyLLlJQl4RAG8n4JLFWZ97yerDYliDGConF8SEJ61kNAXXfBFlT40sHq1atGzMpKjjeLGwewTmpzR6Pn2+XPbvyuPt8RoVAnJ60VnNdCpSpBSr4Gt3evnPZ3v5SGULrsOLn7S+3xhxI4OJB86KrAaNb5uYFUUHz+aOnt6Spe5LmJOMcVbdruEad6Ksh98LTgFlcKRJ2KVIK96zGJ7IUdkdJSBaYA5I3RpXoZ2Vh8BP4Hu4/KwLtZziam9o7mjn9IYpXhBPIt0AAIPLnM4v0b7PsXzuZO7qaOFvZ2/3L80jX6LJb5MFq22FHyNSIIWBSWJDF6qnUqwqkKCkS0lOANFGoba3KikVvviYT5eDBO4vzjMkdnB1z1aNs+TgpKdg3ADnDPGhxvf28jcoiU0VxDyn2vujrCJLMHXz2WpqmpfgqYjr02ibJVqt5OmZ0fhWbN8Td+Ifp9KVPdwipI2yQncJWSJL6fDIJdOC0w3z0sm1/IGtD1nLCX5DbXXKEuG21AGz0hY06Bf24WFSzyvhSIFwiV/Q2btcHQ9yGj3qft3l/wVGE2lv4ItJA3ntwJ+DEwrYleHnMNNxaqJ3zJjOU7Y7roavziA7SD2MLZxyDDAuKgeVf6Ib3yzPssQndZ7r6lhTKS61WFE4psLHHdsa/VW8RHsZB6yJjjUTp8/juNl74UQNHIJuB74eagPVRo3LWICttzupMa7JWhEW59fhGHFPZuHUoY+nMi+Llwy8C9bhq+H4yn8qRuNt18QEsx/4JAHV5POd23wEcOiQyEH0UTpoe2nVTC8GgejWKZbU6S/w5fUUksr9lMCHSK7k2fZjVnJT3awQO3elTMQAqIUaEBZzIqS7zkUypP4y1t1VhifyAw6GGyNomoRmKghdvpKr4sTe9VeR0IeDyC/9mrfAF1wopsXZly9fSPc+MHeOMAKuLEF4DH+yqQ6Py5kCjnQh4PHpUuK1KB/D3amFtUcL+9R7/2UUBxH+rFlN7CMTEzszpX6USEY0M+H3Z20jKCODPIvP4Gb+y6jLPQMoyXM7uzjKxyiqIvFjBNxLBuACNT1y6E8pxeNA5zUPxcDHChTntQ5wjZBEg8xajO0YAdDPz5ut8yzT8YPNi75IFF8FZ2zqPQIyaTj5oq2Fu/xyHaBKIuv8IpoC1o2X4ij6fy1CNNNak4dXz1a3lwi5DctokBgtbsvzXBbTkOQss1JNkjj1AT3IHtonmvvxiAVSaiK6DNxRKiYNoteERRh9pMeSLExgMKFwndKXJnFvB8mdAvogcEpM7BiE+x7v0KjI+hby7QmareQCF/6WJJxx/5SnPG6hp4aTcXhaBXcCtNBJx02HOfFvyqQvMl4ZYCxPXX47gsHtXINPhfeDn3YMVWd4cP29ENUUgAhHgn43H3/GbDfnX0v/fNPk38H6GL+J/LJTfyf40Uf7CJ8Q/LJX/Rob8G1kV9v+dvr+z/Xn+/oUp7D9P4/9Gh/YbXQ/e31vb74x/Nre/sIf3z63uz13+LPq/QEL4DxL4c4s/BfAXVAj/Lgc1RWSUX1n0nx/OTz43ol9//wO2oZUXmxYAAA==\"}")
for name, data in BLOB.items():
    p = ROOT / name; p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(gzip.decompress(base64.b64decode(data)))
sys.path.insert(0, str(ROOT))
import importlib, rollouts as R; importlib.reload(R)
print("unpacked", len(BLOB), "files")

In [ ]:
# 6. Doctor: every line must be ✅ before a big run
import mujoco, numpy as np, time
CHECKS = []
def check(label, ok, detail=""):
    CHECKS.append(bool(ok)); print(("✅ " if ok else "❌ ") + label + (f"  ({detail})" if detail else ""))
import jax
devs = jax.devices()
check("JAX sees a GPU", any(d.platform == "gpu" for d in devs), ", ".join(map(str, devs)))
check("MuJoCo version is the bridge's", mujoco.__version__ == MUJOCO_VERSION, mujoco.__version__)
from mujoco import mjx
src = pathlib.Path(PACKAGE_PATH) if PACKAGE_PATH else ROOT / "sample/contact-arm-experiment.zip"
PKG = R.load_package(src.read_bytes())
check("input package reads and is consistent", True, f"{src.name}: target {PKG['target']}")
m = mujoco.MjModel.from_xml_string(PKG["mjcf"])
check("integrator supported by MJX", m.opt.integrator in (0, 3), f"integrator={m.opt.integrator} (implicitfast=3)")
# One world on CPU (float64, exact browser order) vs one world on MJX (float32 by default).
base = R.Variant(PKG["recipe"], PKG["target"])
ref = R.cpu_reference(base, 2500)
res, secs = R.timed(R.mjx_rollout, [base], 2500)
err = float(np.max(np.abs(res["finalQpos"][0] - np.array(ref["finalQpos"]))))
check("MJX matches native MuJoCo on the package's own trial", err < 5e-3, f"max |Δq| = {err:.2e} rad after 2500 steps; {secs:.1f} s incl. compile")
check("contact seen in both", (res["peakContacts"][0] > 0) == (ref["peakContacts"] > 0), f"mjx {res['peakContacts'][0]}, cpu {ref['peakContacts']}")
# Throughput estimate for the chosen profile (second call is compiled).
VARS = R.sweep(PKG["recipe"], PKG["target"], PROFILE)
probe = VARS[: min(len(VARS), 1024)]
_, secs = R.timed(R.mjx_rollout, probe, 200)
rate = len(probe) * 200 / secs
eta = len(VARS) * R.PROFILES[PROFILE]["steps"] / rate
check("throughput measured", rate > 0, f"{rate:,.0f} world-steps/s -> {len(VARS):,} worlds x {R.PROFILES[PROFILE]['steps']} steps ≈ {eta/60:.1f} min")
print("DOCTOR:", "ALL ✅" if all(CHECKS) else "fix the ❌ lines first")

In [ ]:
# 7. The sweep (resume note: re-running repeats it; results are only written in cell 8)
assert all(CHECKS), "doctor has ❌ lines"
RES, SECS = R.timed(R.mjx_rollout, VARS, R.PROFILES[PROFILE]["steps"])
print(f"{len(VARS):,} worlds in {SECS:.1f} s; finite: {RES['finite'].mean():.1%}; touching: {(RES['peakTouchN'] > 0).mean():.1%}")

In [ ]:
# 8. Write packages: results package + TOP_K variant packages (highest peak touch force), and a summary
import json, datetime
OUT = ROOT / f"out-{RUN_NAME}-{PROFILE}"; OUT.mkdir(exist_ok=True)
meta = {"backend": BACKEND, "mujoco": mujoco.__version__, "jax": jax.__version__, "profile": PROFILE, "steps": R.PROFILES[PROFILE]["steps"],
        "seconds": SECS, "controller": "arm-joint-pd-v1", "precision": "float32 (MJX default)",
        "input": {"name": src.name, "recipe": PKG["recipe"], "target": PKG["target"]},
        "createdAt": datetime.datetime.now(datetime.timezone.utc).isoformat()}
(OUT / "results.zip").write_bytes(R.results_package(PKG, VARS, RES, meta))
order = np.argsort(-RES["peakTouchN"])[:TOP_K]
for rank, i in enumerate(order, 1):
    row = {"rank": rank, "peakTouchN": float(RES["peakTouchN"][i]), "finalQpos": RES["finalQpos"][i].tolist(), **{"meta": meta}}
    (OUT / f"variant-{rank:02d}.zip").write_bytes(R.variant_package(PKG, VARS[i], row))
(OUT / "summary.json").write_text(json.dumps(R.jsonable({**meta, "worlds": len(VARS), "topK": [int(i) for i in order]}), indent=2))
if DRIVE_OUT:
    for f in OUT.iterdir(): shutil.copy2(f, DRIVE_OUT / f.name)
print("wrote", sorted(p.name for p in OUT.iterdir()), "->", DRIVE_OUT or OUT)

In [ ]:
# 9. Done
print("DONE. Tell a hosted model the results are in Drive. Then Runtime > Disconnect and delete runtime to stop spending compute units.")